<img src="./assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

## Project: Deep Learning - Building an Image Classifier for Detecting Plant Diseases

---

## Problem Statement 

Can plant disease be identified by a Convolutional Neural Network model trained on images to aid farmers in preventing the loss of crops?
The simplest approach will be to develop an image classification model broken into categories of known diseases to train the model on. This is to aid the farmer in knowing what treatment will best prevent the loss of each specific type of crop.
The baseline for training will be images of healthy plants, though the Neural Net will not be aware as it will organize images purely on similarity of features.

Further we could refine the model, collect more data to identify disease at early stages if the present model proves useful.

We will explore this question through the use of a Pytorch Resnet 18 model.
Thanks to a great dataset we will be able to categorically sort the images for a more precise insight into the cause of the plant's disease state.

___

#### Import libraries

You must use `pytorch` for this task. Import any modules you'll need below:

In [29]:
# !pip install transformers --upgrade
# !pip install --upgrade xformers
# !pip install torch torchvision
# !pip install pytorch

In [30]:
# !pip install kaggle

In [31]:
# !pip install kagglehub

In [43]:
# imports
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torch.nn.functional as F
import torchvision.transforms as transforms
import seaborn as sns

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models

#### Get the data

You can either download it to your machine and train a model there, or (recommended) connect the Kaggle data with a Google Colab notebook and train the model in the cloud, using a free GPU.  This will be _much_ quicker than running it locally on your machine.

**a. To use Google Colab**:
1. [Open Colab](https://colab.research.google.com/)
2. File -> Upload notebook
3. Upload this notebook
4. Switch to using a GPU **Change runtime type** and select `T4 GPU`

Get the data:
1. [Go to the data on Kaggle](https://www.kaggle.com/datasets/mohitsingh1804/plantvillage?resource=download)
2. Click on Download, select KaggleHub* and copy the code
3. Paste that code into this notebook and run it

*KaggleHub is a Python package that provides a simple API to access Kaggle resources anywhere, including Colab.


Want to know more about Colab?  See [Google’s Colab site](https://colab.google/) for more details or have a look at the [FAQs](https://research.google.com/colaboratory/faq.html).

In [7]:
# Put code for getting data from Kaggle here
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")

print("Path to dataset files:", path)

Path to dataset files: /Users/augustvollbrecht/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1


In [8]:
print(os.listdir(path))

['PlantVillage']


In [9]:
# loading data into train and test

df_train = datasets.ImageFolder(root=os.path.join(path, "PlantVillage", "train"))
df_test = datasets.ImageFolder(root=os.path.join(path, "PlantVillage", "val"))

In [10]:
df_test

Dataset ImageFolder
    Number of datapoints: 10861
    Root location: /Users/augustvollbrecht/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1/PlantVillage/val

In [11]:
df_train

Dataset ImageFolder
    Number of datapoints: 43444
    Root location: /Users/augustvollbrecht/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1/PlantVillage/train

In [12]:
df_test.classes

['Apple___Apple_scab',
 'Apple___Black_rot',
 'Apple___Cedar_apple_rust',
 'Apple___healthy',
 'Blueberry___healthy',
 'Cherry_(including_sour)___Powdery_mildew',
 'Cherry_(including_sour)___healthy',
 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
 'Corn_(maize)___Common_rust_',
 'Corn_(maize)___Northern_Leaf_Blight',
 'Corn_(maize)___healthy',
 'Grape___Black_rot',
 'Grape___Esca_(Black_Measles)',
 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
 'Grape___healthy',
 'Orange___Haunglongbing_(Citrus_greening)',
 'Peach___Bacterial_spot',
 'Peach___healthy',
 'Pepper,_bell___Bacterial_spot',
 'Pepper,_bell___healthy',
 'Potato___Early_blight',
 'Potato___Late_blight',
 'Potato___healthy',
 'Raspberry___healthy',
 'Soybean___healthy',
 'Squash___Powdery_mildew',
 'Strawberry___Leaf_scorch',
 'Strawberry___healthy',
 'Tomato___Bacterial_spot',
 'Tomato___Early_blight',
 'Tomato___Late_blight',
 'Tomato___Leaf_Mold',
 'Tomato___Septoria_leaf_spot',
 'Tomato___Spider_mites Two-spotted_

In [13]:
#  finding out number of classes for model
len(df_train.classes)

38

In [45]:
class_list_train = list(df_train.classes)

In [39]:
class_list = list(df_test.classes)

In [ ]:
#  # of obsercatiosn/samples
# df_train.samples

In [ ]:
# df_test.samples

Your notebook and data are now ready to use.

### 2. Data Preprocessing and Transformation

* Resize all images to 224x224 pixels to match the input size expected by the **ResNet18** model that we'll be using.

* Apply random horizontal flipping to augment the data. Adding random horizontal flips helps generalize the model to real-world variations, such as flipped leaf images.

* Convert the images to PyTorch tensors.

* Normalize the image pixel values to match the pre-trained **ResNet**'s expected input distribution.

Hint: You can load the images and perform all these transformations easily with the `transform` and `ImageFolder` functions in `pytorch`.

In [14]:
# Set parameters for transformation, converting to tensors, normalizing, flipping

tf = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.RandomHorizontalFlip(p=0.5), # Applying a random horizontal flip with 50% probability
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


In [15]:
df_train = datasets.ImageFolder(root=os.path.join(path, "PlantVillage", "train"), transform= tf)
df_test = datasets.ImageFolder(root=os.path.join(path, "PlantVillage", "val"), transform= tf)

When training deep learning models, **batching** and **shuffling** help with computational efficiency and generalization.

#### Batching

Batching refers to dividing the data set into smaller subsets (or "batches") instead of processing the entire data set at once. This improves computational efficiency and training speed.

For example, if the data set has 1,000 samples and the batch size is 32, then:

- **1st Batch**: Samples 1–32
- **2nd Batch**: Samples 33–64
- ...
- **Last Batch (number 32)**: Samples 993–1,000

Each batch is passed through the model independently. After processing a batch, the gradients are computed and the model weights are updated. This process is repeated for every batch in an **epoch** (one complete iteration over the entire data set).

In `pytorch`, the `DataLoader` class automatically divides the data set into batches. The batch size (e.g., batch_size=32) specifies how many samples are in each batch.

#### Shuffling

Shuffling refers to randomly reordering the data set at the start of each epoch. This ensures that the model does not learn any unintended patterns or biases caused by the order of the data.

When `shuffle=True` is specified in the `DataLoader` class, the data set is randomly reordered before creating batches.

Use the `DataLoader` class from `pytorch` to:
* Divide the training and validation sets into batches of 32 images
* Shuffle the training set only

In [16]:
# Set Dataloaders for batching and shuffling

train = DataLoader(df_train, batch_size=32, shuffle=True)
test = DataLoader(df_test, batch_size=32, shuffle=False)

### 3. Initialise the Model

Now our data is ready, it's time to initialise, or build, our model.

We'll be using a pre-trained **ResNet18** model. **ResNet18** is a popular convolutional neural network (CNN) with 18 layers, pre-trained on [ImageNet](https://www.image-net.org/).  (For more information on ImageNet, see [here](https://en.wikipedia.org/wiki/ImageNet).)

Pytorch provides a [number of classification models](https://docs.pytorch.org/vision/main/models.html#classification) for you to explore and use.  You can use them with or without pre-trained weights.

Pre-trained models save time and leverage transfer learning, making them ideal for image classification tasks with limited data sets.

In [17]:
# ResNet-18 with pre-trained ImageNet weights
# This will download the weights if not already present
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/augustvollbrecht/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 55.5MB/s]


`ResNet18`'s original output layer has 1000 classes (corresponding to the 1000 classes in ImageNet).

You should replace the output layer with a layer matching the number of classes in **PlantVillage**.

In [19]:
# setting classes to 38 to match data set
model.fc = nn.Linear(model.fc.in_features, 38)

#### Define the loss function

You should use a cross entropy loss function, which is the standard choice for multi-class classification problems.

In [20]:
criterion = nn.CrossEntropyLoss()

### 4. Train the Model

Train the model for a fixed number of epochs.

For each epoch, iterate through the training data, calculate loss, and update model weights.

In [21]:
# number of epochs
num_epochs = 10
batch_size = 32

In [22]:
# Define the device that the model will train on. 'cuda' means GPU and CPU means CPU.
device = torch.device("cpu")
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Define the optimizer.
We will use the Adam optimizer on the final layer of the model .

In [23]:
# Initializing optimizer
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [24]:
# Training loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    # Training phase
    model.train()
    for inputs, labels in train:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Epoch 9/10
Epoch 10/10


### 5. Evaluate the Model (model validation)

Write a loop to evaluate model performance on the validation set.

Remember, the validation step does not update weights; it only measures the model's classification accuracy on new/unseen data.

Report the overall model accuracy.

In [25]:
# Set model to evaluation mode
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [26]:
# Preped validation data loader/transformed on import 
test = DataLoader(df_test, batch_size=32, shuffle=False)

In [28]:
# Validation loop

correct = 0
total = 0

with torch.no_grad():  # Disable gradient computation
    for data in test:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the model on the validation images: {100 * correct / total:.2f}%')

# Remember to switch back to training mode if you continue training
# model.train()


Accuracy of the model on the validation images: 98.94%


### 6. Conclusions and Recommendations

### With more support for this reasearch we could fine tune the models parameters and dataset imagery to optimizer for wider disease recognition since accuracey seems high enough for in field testing. 

#### Accuracy bar chart:

Could be helpful in identifying which images (not sure how to implement) 
had the higest error rate to focus on when implementing GRAD-Cam, if i am undertaind ing correctley.

In [ ]:
# GRAD-Cam really want to use this but it seems like it would need to be built into the validation model?
read this whole article but couldn't begin to implement: 

See Readme

**I have no idea how to make this more advanced or pull more insights out of the model. 
Hank would you be open to reviewing this or sharing some lessons that could help? 
The models accuracy dostn matter if i dont know whow to use it**

In [ ]:
# Attempts

# Hook: capture activations
activations = {}
def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks for desired layers (e.g., the first conv layer of a block)
model.layer1[0].conv1.register_forward_hook(get_activation('layer1_0_conv1'))
model.layer4[0].conv1.register_forward_hook(get_activation('layer4_0_conv1'))